In [4]:
from google.colab import files
read = files.upload()

Saving Data1-3.xlsx to Data1-3.xlsx
Saving Data1-4.xlsx to Data1-4 (1).xlsx


In [5]:
"""
Bar Graph (ANOVA) + Scatter Matrix + PCA Biplot (overlap fixed)
For: Data1-4.xlsx and Data1-3.xlsx
Sheets: Sheet3=Bulk Density, Sheet4=Soil Moisture, Sheet10=Water Holding Capacity
Groups: FL, GL, CL, DL, C  |  Seasons: Summer, Rainy, Winter

Run: python all_graphs.py
Output: 6 PNG files (3 plots × 2 files)
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Ellipse
from scipy import stats
from scipy.stats import f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

# ═══════════════════════════════════════════════════════════════
# CONFIG
# ═══════════════════════════════════════════════════════════════
FILES = {
    'Data1-4': 'Data1-4.xlsx',
    'Data1-3': 'Data1-3.xlsx',
}

group_order   = ['FL', 'GL', 'CL', 'DL', 'C']
season_order  = ['Summer', 'Rainy', 'Winter']
grp_colors    = {'FL':'#E07B39','GL':'#4A7C59','CL':'#3A7AB5','DL':'#7b5ea7','C':'#c0424e'}
grp_markers   = {'FL':'o','GL':'s','CL':'^','DL':'D','C':'P'}
season_colors = {'Summer':'#E07B39','Rainy':'#3A7AB5','Winter':'#4A7C59'}
params_info   = [('BD','Bulk Density (g/cm³)'),
                 ('SMC','Soil Moisture Content (%)'),
                 ('WHC','Water Holding Capacity (%)')]

# ═══════════════════════════════════════════════════════════════
# HELPERS
# ═══════════════════════════════════════════════════════════════
def extract_long(sheet, id_col, val_cols, param_name):
    mask = sheet[id_col].astype(str).str.strip().str.match(r'^(FL|GL|CL|DL|C)-\d+$')
    rows = []
    for _, row in sheet[mask].iterrows():
        sid = str(row[id_col]).strip()
        grp = sid.split('-')[0]
        for col in val_cols:
            v = row[col]
            if pd.notna(v):
                rows.append({'Group': grp, 'SampleID': sid, param_name: v})
    return pd.DataFrame(rows)

def load_file(filepath):
    sheets = pd.read_excel(filepath, sheet_name=None)
    s3, s4, s10 = sheets['Sheet3'], sheets['Sheet4'], sheets['Sheet10']
    dfs = {}
    cfg = [('Summer','Summer','','2','3'),
           ('Rainy','Rainy','.1','7','8'),
           ('Winter','Winter','.2','12','13')]
    for season, idcol, sfx, u1, u2 in cfg:
        bd = extract_long(s3,  idcol, [f'Bulk Density Values (g/cm3){sfx}', f'Unnamed: {u1}', f'Unnamed: {u2}'], 'BD')
        sm = extract_long(s4,  idcol, [f'Soil moisture content (%){sfx}',   f'Unnamed: {u1}', f'Unnamed: {u2}'], 'SMC')
        wh = extract_long(s10, idcol, [f'Water holding capacity (%){sfx}',  f'Unnamed: {u1}', f'Unnamed: {u2}'], 'WHC')
        for d in [bd, sm, wh]: d['Season'] = season
        merged = bd.merge(sm[['SampleID','Season','SMC']], on=['SampleID','Season'])\
                   .merge(wh[['SampleID','Season','WHC']], on=['SampleID','Season'])
        dfs[season] = merged
    return pd.concat(dfs.values(), ignore_index=True)

def sig_letters(groups_data):
    keys = list(groups_data.keys())
    n_comp = max(len(keys)*(len(keys)-1)//2, 1)
    letters, assigned, alpha_list, letter_idx = {}, {}, ['a','b','c','d','e'], 0
    for k in keys:
        found = False
        for prev_k, let in assigned.items():
            try:
                _, p = stats.ttest_ind(groups_data[k], groups_data[prev_k])
                if p * n_comp > 0.05:
                    letters[k] = let; found = True; break
            except: pass
        if not found:
            letters[k] = alpha_list[min(letter_idx, 4)]
            assigned[k] = letters[k]; letter_idx += 1
    return letters

def confidence_ellipse(x, y, ax, n_std=1.8, **kw):
    if len(x) < 3: return
    cov = np.cov(x, y)
    if np.any(np.isnan(cov)): return
    pearson = cov[0,1] / np.sqrt(cov[0,0]*cov[1,1]+1e-10)
    rx, ry = np.sqrt(max(1+pearson,0))*n_std, np.sqrt(max(1-pearson,0))*n_std
    w, h = rx*2*np.sqrt(cov[0,0]), ry*2*np.sqrt(cov[1,1])
    angle = np.degrees(np.arctan2(cov[0,1], cov[0,0]-cov[1,1])/2)
    ax.add_patch(Ellipse((np.mean(x), np.mean(y)), width=w, height=h, angle=angle, **kw))

def fix_label_overlap(label_positions, min_dist=0.55, iterations=50):
    """Push overlapping labels apart using repulsion algorithm."""
    for _ in range(iterations):
        for i in range(len(label_positions)):
            for j in range(i+1, len(label_positions)):
                dx = label_positions[i][0] - label_positions[j][0]
                dy = label_positions[i][1] - label_positions[j][1]
                dist = np.sqrt(dx**2 + dy**2)
                if dist < min_dist:
                    push = (min_dist - dist) / 2
                    if dist < 1e-6:
                        dx, dy = np.random.randn(), np.random.randn()
                        dist = np.sqrt(dx**2 + dy**2)
                    label_positions[i][0] += push * dx / dist
                    label_positions[i][1] += push * dy / dist
                    label_positions[j][0] -= push * dx / dist
                    label_positions[j][1] -= push * dy / dist
    return label_positions

# Short label names for PCA
short_names = {
    'BD_Summer':'BD\nSum',  'BD_Rainy':'BD\nRain',  'BD_Winter':'BD\nWin',
    'SMC_Summer':'SMC\nSum','SMC_Rainy':'SMC\nRain','SMC_Winter':'SMC\nWin',
    'WHC_Summer':'WHC\nSum','WHC_Rainy':'WHC\nRain','WHC_Winter':'WHC\nWin',
}

# ═══════════════════════════════════════════════════════════════
# MAIN LOOP
# ═══════════════════════════════════════════════════════════════
for fname, fpath in FILES.items():
    print(f"\nProcessing {fname} ...")
    full = load_file(fpath)
    sample_mean = full.groupby(['Group','SampleID','Season'])[['BD','SMC','WHC']].mean().reset_index()

    # ── 1. BAR + ANOVA ──────────────────────────────────────────
    fig, axes = plt.subplots(1, 3, figsize=(20, 7))
    fig.patch.set_facecolor('#fafaf7')
    for ax, (col, title) in zip(axes, params_info):
        ax.set_facecolor('#fafaf7')
        x = np.arange(len(group_order)); w = 0.22
        for i, season in enumerate(season_order):
            sub = full[full['Season']==season]
            means, sems, grp_data = [], [], {}
            for g in group_order:
                vals = sub[sub['Group']==g][col].values
                grp_data[g] = vals
                means.append(vals.mean()); sems.append(vals.std()/np.sqrt(len(vals)))
            letters = sig_letters(grp_data)
            bars = ax.bar(x+i*w, means, w, yerr=sems, capsize=4,
                          color=season_colors[season], edgecolor='white', lw=0.8,
                          alpha=0.9, error_kw=dict(ecolor='#555', lw=1.2))
            for j, (bar, g) in enumerate(zip(bars, group_order)):
                ax.text(bar.get_x()+bar.get_width()/2,
                        bar.get_height()+sems[j]+0.002*bar.get_height(),
                        letters[g], ha='center', va='bottom', fontsize=8,
                        color=season_colors[season], fontweight='bold')
        for j, g in enumerate(group_order):
            gs = [full[(full['Group']==g)&(full['Season']==s)][col].values for s in season_order]
            try:
                _, p = f_oneway(*gs)
                sig = '***' if p<0.001 else '**' if p<0.01 else '*' if p<0.05 else 'ns'
            except: sig = 'ns'
            ax.text(x[j]+w, max(v.mean() for v in gs)*1.15, sig, ha='center', fontsize=8, color='#333')
        ax.set_xticks(x+w); ax.set_xticklabels(group_order, fontsize=12, fontweight='bold')
        ax.set_title(title, fontsize=12, fontweight='bold', pad=8)
        ax.set_ylabel(title, fontsize=10)
        ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4); ax.set_axisbelow(True)
    patches = [mpatches.Patch(color=season_colors[s], label=s) for s in season_order]
    fig.legend(handles=patches, loc='lower center', ncol=3, fontsize=11,
               frameon=False, bbox_to_anchor=(0.5, -0.03))
    fig.suptitle(f'One-Way ANOVA — Soil Properties by Land Use Group & Season ({fname})\n'
                 '(Letters=significance groups; * p<0.05, ** p<0.01, *** p<0.001)',
                 fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.savefig(f'{fname}_bar_anova.png', dpi=150, bbox_inches='tight', facecolor='#fafaf7')
    plt.close(); print(f"  {fname}_bar_anova.png saved")

    # ── 2. SCATTER MATRIX ───────────────────────────────────────
    vars_   = ['BD','SMC','WHC']
    vlabels = ['Bulk Density\n(g/cm³)','Soil Moisture\nContent (%)','Water Holding\nCapacity (%)']
    n = len(vars_)
    fig2, axes2 = plt.subplots(n, n, figsize=(12, 10))
    fig2.patch.set_facecolor('white')
    for i in range(n):
        for j in range(n):
            ax = axes2[i][j]
            ax.set_facecolor('#fafafa')
            for sp in ax.spines.values(): sp.set_color('#cccccc')
            ax.tick_params(labelsize=7, colors='#444')
            if i == j:
                for grp in group_order:
                    vals = sample_mean[sample_mean['Group']==grp][vars_[i]].values
                    ax.hist(vals, bins=8, color=grp_colors[grp], alpha=0.5, density=True, linewidth=0)
                ax.text(0.5, 0.88, vars_[i], transform=ax.transAxes,
                        ha='center', va='top', fontsize=11, fontweight='bold', color='#222')
            else:
                for grp in group_order:
                    sub = sample_mean[sample_mean['Group']==grp]
                    ax.scatter(sub[vars_[j]], sub[vars_[i]], c=grp_colors[grp],
                               marker=grp_markers[grp], s=35, alpha=0.85,
                               edgecolors='white', linewidths=0.4, zorder=3)
                try:
                    m, b, r, p, _ = stats.linregress(sample_mean[vars_[j]].values, sample_mean[vars_[i]].values)
                    xline = np.linspace(sample_mean[vars_[j]].min(), sample_mean[vars_[j]].max(), 100)
                    lc = '#cc3333' if p < 0.05 else '#aaaaaa'
                    ax.plot(xline, m*xline+b, color=lc, lw=1.2, alpha=0.8)
                    pstr = 'p<0.001' if p<0.001 else f'p={p:.3f}'
                    ax.text(0.97, 0.04, f'r={r:.2f}\n{pstr}', transform=ax.transAxes,
                            ha='right', va='bottom', fontsize=6.5, color=lc)
                except: pass
            if i == n-1: ax.set_xlabel(vlabels[j], fontsize=8, labelpad=4)
            if j == 0:   ax.set_ylabel(vlabels[i], fontsize=8, labelpad=4)
    patches2 = [mpatches.Patch(color=grp_colors[g], label=g) for g in group_order]
    fig2.legend(handles=patches2, loc='upper right', ncol=1, fontsize=9,
                title='Land Use\nGroup', title_fontsize=9,
                frameon=True, framealpha=0.9, bbox_to_anchor=(0.99, 0.99))
    fig2.suptitle(f'Scatter Matrix — Soil Properties by Land Use Group ({fname})',
                  fontsize=13, fontweight='bold', y=1.01)
    plt.tight_layout(rect=[0, 0, 0.88, 1])
    plt.savefig(f'{fname}_scatter_matrix.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(); print(f"  {fname}_scatter_matrix.png saved")

    # ── 3. PCA BIPLOT (overlap fixed) ───────────────────────────
    pivot = sample_mean.pivot_table(index=['Group','SampleID'], columns='Season',
                                    values=['BD','SMC','WHC'])
    pivot.columns = [f'{v}_{s}' for v,s in pivot.columns]
    pivot = pivot.dropna().reset_index()
    feature_cols = [c for c in pivot.columns if c not in ['Group','SampleID']]

    X = StandardScaler().fit_transform(pivot[feature_cols])
    pca = PCA(n_components=2)
    scores = pca.fit_transform(X)
    loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
    var1, var2 = pca.explained_variance_ratio_ * 100
    pivot['PC1'], pivot['PC2'] = scores[:,0], scores[:,1]

    fig3, ax3 = plt.subplots(figsize=(11, 9))
    fig3.patch.set_facecolor('white')
    ax3.set_facecolor('#f9f9f9')

    # Ellipses
    for grp in group_order:
        sub = pivot[pivot['Group']==grp]
        confidence_ellipse(sub['PC1'].values, sub['PC2'].values, ax3,
                           edgecolor=grp_colors[grp], facecolor=grp_colors[grp],
                           alpha=0.10, linewidth=1.8, linestyle='--', zorder=1)
    # Points
    for grp in group_order:
        sub = pivot[pivot['Group']==grp]
        ax3.scatter(sub['PC1'], sub['PC2'], c=grp_colors[grp],
                    marker=grp_markers[grp], s=90, label=grp,
                    zorder=5, edgecolors='white', linewidths=0.8)

    # Initial label positions (beyond arrow tips)
    lx_all, ly_all = loadings[:, 0], loadings[:, 1]
    label_positions = [[lx_all[i]*1.22, ly_all[i]*1.22, feature_cols[i]]
                       for i in range(len(feature_cols))]

    # Fix overlaps with repulsion
    label_positions = fix_label_overlap(label_positions, min_dist=0.55, iterations=50)

    # Draw arrows + labels
    arrow_props = dict(arrowstyle='->', color='#cc2222', lw=1.8, mutation_scale=14)
    for i, feat in enumerate(feature_cols):
        lx, ly = lx_all[i], ly_all[i]
        label_x, label_y, _ = label_positions[i]

        # Arrow from origin to loading tip
        ax3.annotate('', xy=(lx, ly), xytext=(0, 0), arrowprops=arrow_props, zorder=6)

        # Dotted connector line if label was pushed far from arrow tip
        if np.sqrt((label_x-lx)**2 + (label_y-ly)**2) > 0.15:
            ax3.plot([lx, label_x], [ly, label_y], color='#cc2222',
                     lw=0.7, ls=':', alpha=0.6, zorder=5)

        # Label with white background to prevent overlap bleed
        ax3.text(label_x, label_y, short_names.get(feat, feat),
                 fontsize=8.5, color='#bb1111', ha='center', va='center',
                 fontweight='bold', zorder=8,
                 bbox=dict(boxstyle='round,pad=0.2', facecolor='white',
                           edgecolor='#ddaaaa', alpha=0.85, linewidth=0.6))

    ax3.axhline(0, color='#bbbbbb', lw=0.8, ls='--', zorder=0)
    ax3.axvline(0, color='#bbbbbb', lw=0.8, ls='--', zorder=0)
    ax3.set_xlabel(f'PC1 ({var1:.1f}% explained var.)', fontsize=12)
    ax3.set_ylabel(f'PC2 ({var2:.1f}% explained var.)', fontsize=12)
    ax3.set_title(f'PCA Biplot — Soil Properties by Land Use Group ({fname})',
                  fontsize=13, fontweight='bold', pad=12)
    ax3.spines['top'].set_visible(False)
    ax3.spines['right'].set_visible(False)

    # Auto-expand axes so labels don't get clipped
    all_x = list(pivot['PC1']) + [p[0] for p in label_positions]
    all_y = list(pivot['PC2']) + [p[1] for p in label_positions]
    pad_x = (max(all_x)-min(all_x)) * 0.15
    pad_y = (max(all_y)-min(all_y)) * 0.15
    ax3.set_xlim(min(all_x)-pad_x, max(all_x)+pad_x)
    ax3.set_ylim(min(all_y)-pad_y, max(all_y)+pad_y)

    legend_elems = [plt.Line2D([0],[0], marker=grp_markers[g], color='w',
                    markerfacecolor=grp_colors[g], markersize=9, label=g) for g in group_order]
    ax3.legend(handles=legend_elems, title='Land Use\nGroup',
               fontsize=10, title_fontsize=10, frameon=True,
               framealpha=0.9, loc='upper left')
    plt.tight_layout()
    plt.savefig(f'{fname}_pca_biplot.png', dpi=150, bbox_inches='tight', facecolor='white')
    plt.close(); print(f"  {fname}_pca_biplot.png saved")

print("\nDone! All 6 graphs generated.")



Processing Data1-4 ...
  Data1-4_bar_anova.png saved
  Data1-4_scatter_matrix.png saved
  Data1-4_pca_biplot.png saved

Processing Data1-3 ...
  Data1-3_bar_anova.png saved
  Data1-3_scatter_matrix.png saved
  Data1-3_pca_biplot.png saved

Done! All 6 graphs generated.


In [6]:
import glob
from google.colab import files

# List all generated PNG files
png_files = glob.glob('*.png')
print(f"Found {len(png_files)} PNG files:\n{png_files}")

# Download each PNG file
for filename in png_files:
    files.download(filename)


Found 6 PNG files:
['Data1-3_pca_biplot.png', 'Data1-3_bar_anova.png', 'Data1-4_bar_anova.png', 'Data1-4_scatter_matrix.png', 'Data1-4_pca_biplot.png', 'Data1-3_scatter_matrix.png']


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>